 Task 1: Data Cleaning & Preprocessing - Titanic Dataset
**Synent Technologies - Data Science Internship** 

**Author:** Gershom Nyamongo Mosora   

**Objective:** Clean and prepare the raw Titanic data so it is ready for analysis.The two steps involved here were cleaning and preprocessing the data and making it ready for analysis.

In [1]:
import pandas as pd
df = pd.read_csv("Titanic.csv")
df.head()

,sex,age,sibsp,parch,fare,embarked,class,who,alone,survived
0,male,22.0,1,0,7.2500,S,Third,man,False,0
1,female,38.0,1,0,71.2833,C,First,woman,False,1
2,female,26.0,0,0,7.9250,S,Third,woman,True,1
3,female,35.0,1,0,53.1000,S,First,woman,False,1
4,male,35.0,0,0,8.0500,S,Third,man,True,0


Checked the shape of the data that i loaded above and it contained 891 rows and 10 colums.
And also before proceeding i ensured that i created a copy of the data using the **copy() function**

In [2]:
df.shape

(891, 10)

In [3]:
df.copy()

,sex,age,sibsp,parch,fare,embarked,class,who,alone,survived
0,male,22.0,1,0,7.2500,S,Third,man,False,0
1,female,38.0,1,0,71.2833,C,First,woman,False,1
2,female,26.0,0,0,7.9250,S,Third,woman,True,1
3,female,35.0,1,0,53.1000,S,First,woman,False,1
4,male,35.0,0,0,8.0500,S,Third,man,True,0
...,...,...,...,...,...,...,...,...,...,...
886,male,27.0,0,0,13.0000,S,Second,man,True,0
887,female,19.0,0,0,30.0000,S,First,woman,True,1
888,female,NaN,1,2,23.4500,S,Third,woman,False,0
889,male,26.0,0,0,30.0000,C,First,man,True,1


In [4]:
raw = df.copy()

**Inspection of the data:** I checked for problems in the data a and noted that age had 177 missing values, embarked 2 and there were 111 duplicated rows which needed to be removed so that they dont affect the values that would be used to fill in the missing data.

In [5]:
print(df.isna().sum())

sex           0
age         177
sibsp         0
parch         0
fare          0
embarked      2
class         0
who           0
alone         0
survived      0
dtype: int64


In [6]:
print("Duplicates:", df.duplicated().sum())

Duplicates: 111


In [7]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df) } rows, {len(df)} remaining")

Removed 111 rows, 780 remaining


In [8]:
raw = pd.read_csv("Titanic.csv")
missing_before = raw.drop_duplicates().isna().sum()
print(missing_before[missing_before > 0])

age         104
embarked      2
dtype: int64


In [9]:
print("Missing now:", df.isna().sum().sum())

Missing now: 106


In [10]:
print("New duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(len(df), "rows remain")

New duplicates: 0
780 rows remain


In [11]:
for col in ["sex", "embarked", "class", "who"]:
    df[col] = df[col].astype("category")
df["survived"] = df["survived"].astype("int8")
df.dtypes

sex         category
age          float64
sibsp          int64
parch          int64
fare         float64
embarked    category
class       category
who         category
alone           bool
survived        int8
dtype: object

There was need to **remane** the colums with appropriate column names which could make sense which also included mapping the port of embarkation letters to their exact full names using the **.map() function** which also made sense since **"S" or "C" or "Q"** had no meaning and could not be understood by a reader who doesnt know what exactly they are representing

In [12]:
df = df.rename(columns={
    "sex": "gender",
    "sibsp": "siblings_spouses",
    "parch": "parents_children",
    "embarked": "port_of_embarkation",
    "class": "passenger_class",
    "who": "passenger_type",
    "alone": "travelling_alone",
})

df["port_of_embarkation"] = (
    df["port_of_embarkation"]
    .map({"S": "Southampton", "C": "Cherbourg", "Q": "Queenstown"})
    .astype("category")
)

print(df.columns.tolist())
df["port_of_embarkation"].value_counts()

['gender', 'age', 'siblings_spouses', 'parents_children', 'fare', 'port_of_embarkation', 'passenger_class', 'passenger_type', 'travelling_alone', 'survived']


port_of_embarkation
Southampton    565
Cherbourg      155
Queenstown      58
Name: count, dtype: int64

In [13]:
print("Missing values:", df.isna().sum().sum())
print("Duplicates:", df.duplicated().sum())
print(df.shape)

df.to_csv("Titanic_clean.csv", index=False)

Missing values: 106
Duplicates: 0
(780, 10)


**Handling Missing values**:This one involved filling the missing values using the **median age of passengers with the same gender and class** and checking again if there are any missing values to esure that all missing values have been handled correctly

In [14]:
df["age"] = df["age"].fillna(
    df.groupby(["gender", "passenger_class"], observed=True)["age"].transform("median")
)
df["port_of_embarkation"] = df["port_of_embarkation"].fillna(
    df["port_of_embarkation"].mode()[0]
)

print("Missing now:", df.isna().sum().sum())

# Duplicates created by filling the ages
print("New duplicates:", df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print(len(df), "rows remain")

Missing now: 0
New duplicates: 8
772 rows remain


**Validated and saved the Data**-This is where now i validated the data to ensure that there is no any other missing value and that there are no duplicates since from above there were 8 duplicates and there was need to validate and removed them and ensured that everything is working well

In [15]:
print("Missing values:", df.isna().sum().sum())
print("Duplicates:", df.duplicated().sum())
print(df.shape)
print(df["port_of_embarkation"].value_counts())

df.to_csv("Titanic_clean.csv", index=False)

Missing values: 0
Duplicates: 0
(772, 10)
port_of_embarkation
Southampton    562
Cherbourg      153
Queenstown      57
Name: count, dtype: int64
